In [1]:
import pandas as pd
import os

In [2]:
pd.set_option('display.max_columns', None)

### Files

See **parameter explanation for detail information

In [3]:
file_path = "../Data/parameters of interest.xlsx"
files = pd.ExcelFile(file_path).sheet_names

In [4]:
files

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'family_hx']

In [5]:
shared_path = "../Data/R3Data"

### 8. patient_demo
<span style="color: blue;">Change idx value **based on EHR file**</span>

In [6]:
idx = 8
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

patient_demo
['PATIENT_STUDY_ID', 'BIRTH_DATE', 'GENDER_TITLE', 'RACE_TITLE', 'ETHNIC_TITLE']


In [7]:
file_names = [
    str(file) + ".txt", 
    str(file) + ".csv",
    str(file) + ".csv"
]
file_names

['patient_demo.txt', 'patient_demo.csv', 'patient_demo.csv']

### Cancer

In [8]:
study = "Cancer"
folder = ["R3_3787_Lee_Cancer_Extract_Files", "R3_3787_Lee_Data_Cancer_20240509", "R3_3787_Lee_Data_Cancer_20250912"]

In [9]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [10]:
final_df

,PATIENT_STUDY_ID,GENDER_TITLE,BIRTH_DATE,DEATH_DATE,RACE_TITLE,ETHNIC_TITLE
0,4335275754,FEMALE,07/01/1943,NaN,DECLINED,NaN
1,4335887037,FEMALE,07/01/1946,NaN,WHITE,NOT HISPANIC OR LATINO
2,4333019261,FEMALE,07/01/1967,NaN,WHITE,DECLINED
3,4333683065,FEMALE,07/01/1975,NaN,WHITE,NOT HISPANIC OR LATINO
4,4335895898,FEMALE,07/01/1977,NaN,WHITE,NOT HISPANIC OR LATINO
...,...,...,...,...,...,...
10795,4334873026,FEMALE,07/01/1964,NaN,"UNREPORTED,CHOSE NOT TO DISCLOSE RACE",NaN
10796,4335275754,FEMALE,07/01/1943,NaN,"UNREPORTED,CHOSE NOT TO DISCLOSE RACE",NaN
10797,4335545232,FEMALE,07/01/1973,NaN,"UNREPORTED,CHOSE NOT TO DISCLOSE RACE",NaN
10798,4336264236,FEMALE,07/01/1973,NaN,"UNREPORTED,CHOSE NOT TO DISCLOSE RACE",NaN


In [11]:
final_df['BIRTH_DATE'] = pd.to_datetime(final_df['BIRTH_DATE'], format='%m/%d/%Y')
final_df['BIRTH_DATE'] = final_df['BIRTH_DATE'].dt.strftime('%Y-%m-%d')

In [12]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'BIRTH_DATE'], ascending=[True, True], inplace=True, ignore_index=True)

#### <span style="color: blue;">Rename columns values </span> 

In [13]:
final_df['ETHNIC_TITLE'].unique()

array(['NON-HISPANIC OR LATINO/A',
       'ANOTHER HISPANIC, LATINO/A, OR SPANISH ORIGIN',
       'UNREPORTED/CHOSE NOT TO DISCLOSE', 'NOT HISPANIC OR LATINO', nan,
       'DECLINED', 'NOT SPECIFIED', 'HISPANIC OR LATINO',
       'MEXICAN, MEXICAN AMERICAN, CHICANO/A', 'PUERTO RICAN'],
      dtype=object)

In [14]:
final_df['RACE_TITLE'].unique()

array(['KOREAN', 'WHITE', 'ASIAN INDIAN', 'BLACK/AFRICAN AMERICAN',
       'UNREPORTED,CHOSE NOT TO DISCLOSE RACE', 'OTHER ASIAN', 'DECLINED',
       'VIETNAMESE', 'CHINESE', 'AMERICAN INDIAN/ALASKA NATIVE', 'BLACK',
       'FILIPINO', 'UNKNOWN', 'OTHER', 'NOT SPECIFIED', 'JAPANESE',
       'INDIAN (ASIAN)', 'AMERICAN INDIAN', 'ALASKA NATIVE', 'HAWAIIAN',
       nan, 'OTHER PACIFIC ISLANDER'], dtype=object)

In [15]:
mapping_dict = {
    'NON-HISPANIC OR LATINO/A': 'NOT HISPANIC OR LATINO',
    'ANOTHER HISPANIC, LATINO/A, OR SPANISH ORIGIN': 'ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN',
    'MEXICAN, MEXICAN AMERICAN, CHICANO/A': 'MEXICAN, MEXICAN AMERICAN, CHICANO',
    'NOT SPECIFIED': 'UNREPORTED/CHOSE NOT TO DISCLOSE',
    'DECLINED': 'UNREPORTED/CHOSE NOT TO DISCLOSE',
    'ASIAN INDIAN': 'INDIAN (ASIAN)',
    'ALASKA NATIVE': 'AMERICAN INDIAN/ALASKA NATIVE',
    'AMERICAN INDIAN': 'AMERICAN INDIAN/ALASKA NATIVE',    
    'BLACK/AFRICAN AMERICAN': 'BLACK',
    'UNKNOWN': 'UNREPORTED/CHOSE NOT TO DISCLOSE',
    'UNREPORTED,CHOSE NOT TO DISCLOSE RACE': 'UNREPORTED/CHOSE NOT TO DISCLOSE',
    'NATIVE HAWAIIAN': 'HAWAIIAN'
}

In [16]:
final_df['ETHNIC_TITLE'] = final_df['ETHNIC_TITLE'].map(mapping_dict).fillna(final_df['ETHNIC_TITLE'])
final_df['RACE_TITLE'] = final_df['RACE_TITLE'].map(mapping_dict).fillna(final_df['RACE_TITLE'])

In [17]:
final_df['ETHNIC_TITLE'].unique()

array(['NOT HISPANIC OR LATINO',
       'ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN',
       'UNREPORTED/CHOSE NOT TO DISCLOSE', nan, 'HISPANIC OR LATINO',
       'MEXICAN, MEXICAN AMERICAN, CHICANO', 'PUERTO RICAN'], dtype=object)

In [18]:
final_df['RACE_TITLE'].unique()

array(['KOREAN', 'WHITE', 'INDIAN (ASIAN)', 'BLACK',
       'UNREPORTED/CHOSE NOT TO DISCLOSE', 'OTHER ASIAN', 'VIETNAMESE',
       'CHINESE', 'AMERICAN INDIAN/ALASKA NATIVE', 'FILIPINO', 'OTHER',
       'JAPANESE', 'HAWAIIAN', nan, 'OTHER PACIFIC ISLANDER'],
      dtype=object)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [19]:
unique_df = final_df[extract_cols].copy()

In [20]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [21]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [22]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [23]:
unique_df[unique_df["PATIENT_STUDY_ID"]==4330018595]

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,RACE_TITLE,ETHNIC_TITLE,duplicate_count
0,4330018595,1965-07-01,FEMALE,KOREAN,NOT HISPANIC OR LATINO,1


In [24]:
unique_df['duplicate_count'].unique()

array([1, 2, 3])

In [25]:
unique_df[extract_cols]

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,RACE_TITLE,ETHNIC_TITLE
0,4330018595,1965-07-01,FEMALE,KOREAN,NOT HISPANIC OR LATINO
1,4330029102,1974-07-01,FEMALE,WHITE,"ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN"
2,4330044371,1975-07-01,FEMALE,WHITE,NOT HISPANIC OR LATINO
3,4330066079,1989-07-01,FEMALE,WHITE,UNREPORTED/CHOSE NOT TO DISCLOSE
4,4330103113,1952-07-01,FEMALE,WHITE,NOT HISPANIC OR LATINO
...,...,...,...,...,...
9319,4339930609,1976-07-01,FEMALE,WHITE,UNREPORTED/CHOSE NOT TO DISCLOSE
9320,4339933706,1947-07-01,FEMALE,WHITE,NaN
9321,4339943307,1973-07-01,FEMALE,VIETNAMESE,NOT HISPANIC OR LATINO
9322,4339945522,1974-07-01,FEMALE,WHITE,NOT HISPANIC OR LATINO


#### <span style="color: blue;">One hot encoding for RACE </span> 

**Before = <span style="color: blue;">df = unique_df[extract_cols]</span> , after  = <span style="color: blue;">df_encoded</span>**

In [26]:
df = unique_df[extract_cols]

In [27]:
grouping_keys = ['PATIENT_STUDY_ID', 'BIRTH_DATE', 'GENDER_TITLE']

In [28]:
df_dummies = pd.get_dummies(
    df,
    columns=['RACE_TITLE'],
    prefix='',
    prefix_sep=''
)

In [29]:
race_cols = ['KOREAN', 'WHITE', 'INDIAN (ASIAN)', 'BLACK',
       'UNREPORTED/CHOSE NOT TO DISCLOSE', 'OTHER ASIAN', 'VIETNAMESE',
       'CHINESE', 'AMERICAN INDIAN/ALASKA NATIVE', 'FILIPINO', 'OTHER',
       'JAPANESE', 'HAWAIIAN', 'OTHER PACIFIC ISLANDER']

In [30]:
len(race_cols)

14

In [31]:
final_columns = grouping_keys + race_cols

In [32]:
df_encoded = df_dummies.groupby(grouping_keys, as_index=False)[race_cols].sum()

In [33]:
df_encoded = df_encoded[final_columns]

In [34]:
df_encoded

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,KOREAN,WHITE,INDIAN (ASIAN),BLACK,UNREPORTED/CHOSE NOT TO DISCLOSE,OTHER ASIAN,VIETNAMESE,CHINESE,AMERICAN INDIAN/ALASKA NATIVE,FILIPINO,OTHER,JAPANESE,HAWAIIAN,OTHER PACIFIC ISLANDER
0,4330018595,1965-07-01,FEMALE,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,4330029102,1974-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
2,4330044371,1975-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
3,4330066079,1989-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4,4330103113,1952-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9263,4339930609,1976-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
9264,4339933706,1947-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
9265,4339943307,1973-07-01,FEMALE,0,0,0,0,0,0,1,0,0,0,0,0,0,0
9266,4339945522,1974-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [35]:
df_encoded[race_cols] = (df_encoded[race_cols] != 0).astype(int)

In [36]:
df_encoded.sort_values(by=['PATIENT_STUDY_ID'], ascending=[True], inplace=True, ignore_index=True)

In [37]:
df_encoded.columns = df_encoded.columns[0:3].tolist() + ['RACE_'+r for r in race_cols]

#### <span style="color: blue;">One hot encoding for ETHNICITY </span> 

**Before = <span style="color: blue;">df = unique_df[extract_cols]</span> , after  = <span style="color: blue;">df_encoded</span>**

In [38]:
df = unique_df[extract_cols]

In [39]:
grouping_keys = ['PATIENT_STUDY_ID', 'BIRTH_DATE', 'GENDER_TITLE']

In [40]:
df_dummies = pd.get_dummies(
    df,
    columns=['ETHNIC_TITLE'],
    prefix='',
    prefix_sep=''
)

In [41]:
ethnic_cols = ['NOT HISPANIC OR LATINO',
       'ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN',
       'UNREPORTED/CHOSE NOT TO DISCLOSE', 'HISPANIC OR LATINO',
       'MEXICAN, MEXICAN AMERICAN, CHICANO', 'PUERTO RICAN']

In [42]:
len(ethnic_cols)

6

In [43]:
final_columns = grouping_keys + ethnic_cols

In [44]:
df_encoded_p2 = df_dummies.groupby(grouping_keys, as_index=False)[ethnic_cols].sum()

In [45]:
df_encoded_p2 = df_encoded_p2[final_columns]

In [46]:
df_encoded_p2

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,NOT HISPANIC OR LATINO,"ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",UNREPORTED/CHOSE NOT TO DISCLOSE,HISPANIC OR LATINO,"MEXICAN, MEXICAN AMERICAN, CHICANO",PUERTO RICAN
0,4330018595,1965-07-01,FEMALE,1,0,0,0,0,0
1,4330029102,1974-07-01,FEMALE,0,1,0,0,0,0
2,4330044371,1975-07-01,FEMALE,1,0,0,0,0,0
3,4330066079,1989-07-01,FEMALE,0,0,1,0,0,0
4,4330103113,1952-07-01,FEMALE,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
9263,4339930609,1976-07-01,FEMALE,0,0,1,0,0,0
9264,4339933706,1947-07-01,FEMALE,0,0,0,0,0,0
9265,4339943307,1973-07-01,FEMALE,1,0,0,0,0,0
9266,4339945522,1974-07-01,FEMALE,1,0,0,0,0,0


In [47]:
df_encoded_p2[ethnic_cols] = (df_encoded_p2[ethnic_cols] != 0).astype(int)

In [48]:
df_encoded_p2.sort_values(by=['PATIENT_STUDY_ID'], ascending=[True], inplace=True, ignore_index=True)

In [49]:
df_encoded_p2.columns = df_encoded_p2.columns[0:3].tolist() + ['ETHNIC_'+r for r in ethnic_cols]

In [50]:
df_final_encoded = pd.concat((df_encoded, df_encoded_p2.drop(columns=['PATIENT_STUDY_ID', 'BIRTH_DATE', 'GENDER_TITLE'])), axis=1)

In [51]:
df_final_encoded

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",ETHNIC_PUERTO RICAN
0,4330018595,1965-07-01,FEMALE,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,4330029102,1974-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2,4330044371,1975-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,4330066079,1989-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
4,4330103113,1952-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9263,4339930609,1976-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
9264,4339933706,1947-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9265,4339943307,1973-07-01,FEMALE,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
9266,4339945522,1974-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


#### <span style="color: orange;">SAVE </span> 

In [52]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df_final_encoded.to_excel(output_file, index=False)

### Control

In [53]:
file_names = [
    str(file) + "_controls.txt", 
    str(file) + "_controls.csv",
]
file_names

['patient_demo_controls.txt', 'patient_demo_controls.csv']

In [54]:
study = "Control"
folder = ["R3_3787_Lee_Control_Extract_Files", "R3_3787_Lee_Data_Controls_20240508"]

In [55]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [56]:
final_df

,PATIENT_STUDY_ID,GENDER_TITLE,BIRTH_DATE,DEATH_DATE,RACE_TITLE,ETHNIC_TITLE
0,4333333535,FEMALE,07/01/1958,NaN,WHITE,NOT HISPANIC OR LATINO
1,4333333026,FEMALE,07/01/1957,NaN,WHITE,NOT HISPANIC OR LATINO
2,4333333813,FEMALE,07/01/1975,NaN,WHITE,HISPANIC OR LATINO
3,4333334342,FEMALE,07/01/1973,NaN,WHITE,NOT HISPANIC OR LATINO
4,4333335240,FEMALE,07/01/1963,NaN,WHITE,NOT HISPANIC OR LATINO
...,...,...,...,...,...,...
49090,4335215861,FEMALE,07/01/1979,NaN,NaN,NaN
49091,4334625770,FEMALE,07/01/1966,NaN,NaN,NON-HISPANIC OR LATINO/A
49092,4335984974,FEMALE,07/01/1966,NaN,NaN,NON-HISPANIC OR LATINO/A
49093,4333204829,FEMALE,07/01/1953,NaN,NaN,NON-HISPANIC OR LATINO/A


In [57]:
final_df['BIRTH_DATE'] = pd.to_datetime(final_df['BIRTH_DATE'], format='%m/%d/%Y')
final_df['BIRTH_DATE'] = final_df['BIRTH_DATE'].dt.strftime('%Y-%m-%d')

In [58]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'BIRTH_DATE'], ascending=[True, True], inplace=True, ignore_index=True)

#### <span style="color: blue;">Rename columns values </span> 

In [59]:
final_df['ETHNIC_TITLE'].unique()

array(['NON-HISPANIC OR LATINO/A', nan, 'NOT HISPANIC OR LATINO',
       'NOT SPECIFIED', 'UNREPORTED/CHOSE NOT TO DISCLOSE',
       'ANOTHER HISPANIC, LATINO/A, OR SPANISH ORIGIN',
       'HISPANIC OR LATINO', 'DECLINED',
       'MEXICAN, MEXICAN AMERICAN, CHICANO/A'], dtype=object)

In [60]:
final_df['RACE_TITLE'].unique()

array(['WHITE', 'NOT SPECIFIED', nan, 'BLACK/AFRICAN AMERICAN',
       'UNREPORTED,CHOSE NOT TO DISCLOSE RACE', 'FILIPINO', 'CHINESE',
       'OTHER ASIAN', 'OTHER', 'JAPANESE', 'OTHER PACIFIC ISLANDER',
       'BLACK', 'DECLINED', 'ASIAN INDIAN', 'VIETNAMESE',
       'AMERICAN INDIAN/ALASKA NATIVE', 'KOREAN', 'INDIAN (ASIAN)',
       'AMERICAN INDIAN', 'UNKNOWN', 'ALASKA NATIVE', 'NATIVE HAWAIIAN'],
      dtype=object)

In [61]:
final_df['ETHNIC_TITLE'] = final_df['ETHNIC_TITLE'].map(mapping_dict).fillna(final_df['ETHNIC_TITLE'])
final_df['RACE_TITLE'] = final_df['RACE_TITLE'].map(mapping_dict).fillna(final_df['RACE_TITLE'])

In [62]:
final_df['ETHNIC_TITLE'].unique()

array(['NOT HISPANIC OR LATINO', nan, 'UNREPORTED/CHOSE NOT TO DISCLOSE',
       'ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN',
       'HISPANIC OR LATINO', 'MEXICAN, MEXICAN AMERICAN, CHICANO'],
      dtype=object)

In [63]:
final_df['RACE_TITLE'].unique()

array(['WHITE', 'UNREPORTED/CHOSE NOT TO DISCLOSE', nan, 'BLACK',
       'FILIPINO', 'CHINESE', 'OTHER ASIAN', 'OTHER', 'JAPANESE',
       'OTHER PACIFIC ISLANDER', 'INDIAN (ASIAN)', 'VIETNAMESE',
       'AMERICAN INDIAN/ALASKA NATIVE', 'KOREAN', 'HAWAIIAN'],
      dtype=object)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [64]:
unique_df = final_df.copy()

In [65]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [66]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [67]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [68]:
unique_df['duplicate_count'].unique()

array([1, 2])

In [69]:
unique_df[unique_extract_cols]

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,RACE_TITLE,ETHNIC_TITLE,duplicate_count
0,4330000534,1973-07-01,FEMALE,WHITE,NOT HISPANIC OR LATINO,1
1,4330002818,1977-07-01,FEMALE,WHITE,NOT HISPANIC OR LATINO,1
2,4330018244,1961-07-01,FEMALE,UNREPORTED/CHOSE NOT TO DISCLOSE,NaN,1
3,4330022845,1959-07-01,FEMALE,WHITE,NOT HISPANIC OR LATINO,1
4,4330023215,1959-07-01,FEMALE,WHITE,NOT HISPANIC OR LATINO,1
...,...,...,...,...,...,...
38559,4339949064,1975-07-01,FEMALE,UNREPORTED/CHOSE NOT TO DISCLOSE,NOT HISPANIC OR LATINO,2
38560,4339967297,1971-07-01,FEMALE,WHITE,NOT HISPANIC OR LATINO,1
38561,4339978786,1953-07-01,FEMALE,WHITE,NOT HISPANIC OR LATINO,1
38562,4339982047,1965-07-01,FEMALE,INDIAN (ASIAN),NOT HISPANIC OR LATINO,1


#### <span style="color: blue;">One hot encoding for RACE </span> 

**Before = <span style="color: blue;">df = unique_df[extract_cols]</span> , after  = <span style="color: blue;">df_encoded</span>**

In [70]:
df = unique_df[extract_cols]

In [71]:
grouping_keys = ['PATIENT_STUDY_ID', 'BIRTH_DATE', 'GENDER_TITLE']

In [72]:
df_dummies = pd.get_dummies(
    df,
    columns=['RACE_TITLE'],
    prefix='',
    prefix_sep=''
)

In [73]:
df['RACE_TITLE'].unique()

array(['WHITE', 'UNREPORTED/CHOSE NOT TO DISCLOSE', nan, 'BLACK',
       'FILIPINO', 'CHINESE', 'OTHER ASIAN', 'OTHER', 'JAPANESE',
       'OTHER PACIFIC ISLANDER', 'INDIAN (ASIAN)', 'VIETNAMESE',
       'AMERICAN INDIAN/ALASKA NATIVE', 'KOREAN', 'HAWAIIAN'],
      dtype=object)

In [74]:
race_cols = ['KOREAN', 'WHITE', 'INDIAN (ASIAN)', 'BLACK',
       'UNREPORTED/CHOSE NOT TO DISCLOSE', 'OTHER ASIAN', 'VIETNAMESE',
       'CHINESE', 'AMERICAN INDIAN/ALASKA NATIVE', 'FILIPINO', 'OTHER',
       'JAPANESE', 'HAWAIIAN', 'OTHER PACIFIC ISLANDER']

In [75]:
len(race_cols)

14

In [76]:
final_columns = grouping_keys + race_cols

In [77]:
df_encoded = df_dummies.groupby(grouping_keys, as_index=False)[race_cols].sum()

In [78]:
df_encoded = df_encoded[final_columns]

In [79]:
df_encoded

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,KOREAN,WHITE,INDIAN (ASIAN),BLACK,UNREPORTED/CHOSE NOT TO DISCLOSE,OTHER ASIAN,VIETNAMESE,CHINESE,AMERICAN INDIAN/ALASKA NATIVE,FILIPINO,OTHER,JAPANESE,HAWAIIAN,OTHER PACIFIC ISLANDER
0,4330000534,1973-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
1,4330002818,1977-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
2,4330018244,1961-07-01,FEMALE,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,4330022845,1959-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4,4330023215,1959-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38334,4339949064,1975-07-01,FEMALE,0,0,0,0,1,0,0,0,0,0,0,0,0,0
38335,4339967297,1971-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
38336,4339978786,1953-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0
38337,4339982047,1965-07-01,FEMALE,0,0,1,0,0,0,0,0,0,0,0,0,0,0


In [80]:
df_encoded[race_cols] = (df_encoded[race_cols] != 0).astype(int)

In [81]:
df_encoded.sort_values(by=['PATIENT_STUDY_ID'], ascending=[True], inplace=True, ignore_index=True)

In [82]:
df_encoded.columns = df_encoded.columns[0:3].tolist() + ['RACE_'+r for r in race_cols]

#### <span style="color: blue;">One hot encoding for ETHNICITY </span> 

**Before = <span style="color: blue;">df = unique_df[extract_cols]</span> , after  = <span style="color: blue;">df_encoded</span>**

In [83]:
df = unique_df[extract_cols]

In [84]:
grouping_keys = ['PATIENT_STUDY_ID', 'BIRTH_DATE', 'GENDER_TITLE']

In [85]:
df_dummies = pd.get_dummies(
    df,
    columns=['ETHNIC_TITLE'],
    prefix='',
    prefix_sep=''
)

In [86]:
df['ETHNIC_TITLE'].unique()

array(['NOT HISPANIC OR LATINO', nan, 'UNREPORTED/CHOSE NOT TO DISCLOSE',
       'ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN',
       'HISPANIC OR LATINO', 'MEXICAN, MEXICAN AMERICAN, CHICANO'],
      dtype=object)

In [87]:
ethnic_cols = ['NOT HISPANIC OR LATINO',
       'ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN',
       'UNREPORTED/CHOSE NOT TO DISCLOSE', 'HISPANIC OR LATINO',
       'MEXICAN, MEXICAN AMERICAN, CHICANO']

In [88]:
len(ethnic_cols)

5

In [89]:
final_columns = grouping_keys + ethnic_cols

In [90]:
df_encoded_p2 = df_dummies.groupby(grouping_keys, as_index=False)[ethnic_cols].sum()

In [91]:
df_encoded_p2 = df_encoded_p2[final_columns]

In [92]:
df_encoded_p2

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,NOT HISPANIC OR LATINO,"ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",UNREPORTED/CHOSE NOT TO DISCLOSE,HISPANIC OR LATINO,"MEXICAN, MEXICAN AMERICAN, CHICANO"
0,4330000534,1973-07-01,FEMALE,1,0,0,0,0
1,4330002818,1977-07-01,FEMALE,1,0,0,0,0
2,4330018244,1961-07-01,FEMALE,0,0,0,0,0
3,4330022845,1959-07-01,FEMALE,1,0,0,0,0
4,4330023215,1959-07-01,FEMALE,1,0,0,0,0
...,...,...,...,...,...,...,...,...
38334,4339949064,1975-07-01,FEMALE,1,0,0,0,0
38335,4339967297,1971-07-01,FEMALE,1,0,0,0,0
38336,4339978786,1953-07-01,FEMALE,1,0,0,0,0
38337,4339982047,1965-07-01,FEMALE,1,0,0,0,0


In [93]:
df_encoded_p2[ethnic_cols] = (df_encoded_p2[ethnic_cols] != 0).astype(int)

In [94]:
df_encoded_p2.sort_values(by=['PATIENT_STUDY_ID'], ascending=[True], inplace=True, ignore_index=True)

In [95]:
df_encoded_p2.columns = df_encoded_p2.columns[0:3].tolist() + ['ETHNIC_'+r for r in ethnic_cols]

In [96]:
df_final_encoded = pd.concat((df_encoded, df_encoded_p2.drop(columns=['PATIENT_STUDY_ID', 'BIRTH_DATE', 'GENDER_TITLE'])), axis=1)

In [97]:
df_final_encoded

,PATIENT_STUDY_ID,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO"
0,4330000534,1973-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
1,4330002818,1977-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2,4330018244,1961-07-01,FEMALE,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4330022845,1959-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,4330023215,1959-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38334,4339949064,1975-07-01,FEMALE,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
38335,4339967297,1971-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
38336,4339978786,1953-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
38337,4339982047,1965-07-01,FEMALE,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0


#### <span style="color: orange;">SAVE </span> 

In [98]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
df_final_encoded.to_excel(output_file, index=False)